# MTMC 11a — WILDTRACK Stages 0-2 (Tracking + ReID Features)
Person multi-camera tracking pipeline on WILDTRACK dataset (7 cameras, 1920x1080, 2fps).

In [ ]:
import os, sys, subprocess, shutil, json, time, tarfile, re
from pathlib import Path

# --- Guard: detect GPU BEFORE importing torch ---
# Kaggle's PyTorch 2.10+ drops P100 (sm_60) support.
# If we got a P100, downgrade to a compatible build first.
if shutil.which("nvidia-smi"):
    _nvsmi = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True)
    if _nvsmi.returncode == 0 and _nvsmi.stdout.strip():
        _gpu_name, _cap = _nvsmi.stdout.strip().split(",", 1)
        _match = re.search(r"(\d+)\.(\d+)", _cap)
        if _match:
            _major, _minor = _match.groups()
            _sm = int(_major) * 10 + int(_minor)
            if _sm < 70:
                print(f"\u26a0 GPU {_gpu_name.strip()} (sm_{_sm}) \u2014 installing compatible PyTorch ...")
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])
                print("\u2713 Compatible PyTorch installed")

import torch

print(f"Python : {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA   : {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  ({p.total_memory/1024**3:.1f} GB)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {DEVICE}")

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Clone repository
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT  = WORK_DIR / "gp"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    print("Repo already present, pulling latest ...")
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(str(PROJECT))
sys.path.insert(0, str(PROJECT))
print(f"\n\u2713 Repo ready at {PROJECT}")

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

try:
    import faiss
    print(f"faiss ok ({faiss.__version__})")
except ImportError:
    try:
        pip("faiss-gpu")
    except Exception:
        pip("faiss-cpu")

try:
    import trackeval
    print("trackeval ok")
except ImportError:
    pip("git+https://github.com/JonathonLuiten/TrackEval.git")

pip(
    "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click",
    "numpy", "scipy", "pandas", "scikit-learn", "ultralytics>=8.1", "boxmot>=10.0",
    "timm>=0.9", "opencv-python-headless", "torchreid", "tqdm", "Pillow"
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
print("\n\u2713 Dependencies installed")

In [ ]:
import shutil, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

# Try both possible mount paths
WEIGHTS_INPUT = None
for candidate in [
    Path("/kaggle/input/datasets/mrkdagods/mtmc-weights"),
    Path("/kaggle/input/mtmc-weights"),
]:
    if candidate.exists():
        WEIGHTS_INPUT = candidate
        break

if WEIGHTS_INPUT is None:
    # List what's available
    inp = Path("/kaggle/input")
    if inp.exists():
        print("Available inputs:")
        for p in sorted(inp.rglob("*"))[:30]:
            print(f"  {p}")
    raise FileNotFoundError("Weights dataset not found - attach mrkdagods/mtmc-weights")

print(f"Weights found at: {WEIGHTS_INPUT}")

# Copy models/ tree
MODELS_DST = PROJECT / "models"
if MODELS_DST.is_symlink():
    MODELS_DST.unlink()
if MODELS_DST.exists():
    shutil.rmtree(MODELS_DST)

print(f"Copying models from {WEIGHTS_INPUT} ...")
shutil.copytree(str(WEIGHTS_INPUT), str(MODELS_DST))

# Verify critical person model exists
person_model = MODELS_DST / "reid" / "person_transreid_vit_base_market1501.pth"
assert person_model.exists(), f"Person ReID model not found at {person_model}"
print(f"Person ReID: {person_model.name} ({person_model.stat().st_size/1024**2:.1f} MB)")

# List all copied models
for f in sorted(MODELS_DST.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(MODELS_DST)} ({f.stat().st_size/1024**2:.1f} MB)")

In [ ]:
import os, sys, subprocess
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

# Find WILDTRACK dataset - try multiple mount patterns
WILDTRACK_INPUT = None
for candidate in [
    Path("/kaggle/input/large-scale-multicamera-detection-dataset"),
    Path("/kaggle/input/datasets/aryashah2k/large-scale-multicamera-detection-dataset"),
    Path("/kaggle/input/wildtrack"),
    Path("/kaggle/input/datasets/aryashah2k/wildtrack"),
]:
    if candidate.exists():
        WILDTRACK_INPUT = candidate
        break

# Fallback: scan all inputs
if WILDTRACK_INPUT is None:
    inp = Path("/kaggle/input")
    if inp.exists():
        print("Scanning /kaggle/input/ for WILDTRACK dataset...")
        for p in sorted(inp.rglob("*")):
            if p.is_dir() and p.name.lower() in ("image_subsets", "wildtrack_dataset", "wildtrack"):
                WILDTRACK_INPUT = p.parent if p.name.lower() != "wildtrack" else p
                print(f"  Found WILDTRACK-like dir: {p}")
                break
            # Also print top-level dirs for debugging
            if p.parent == inp or (p.parent.parent == inp and p.parent.name == "datasets"):
                if p.is_dir():
                    print(f"  {p}")

if WILDTRACK_INPUT is None:
    raise FileNotFoundError("WILDTRACK dataset not found in /kaggle/input/ - check dataset attachment")

print(f"WILDTRACK dataset: {WILDTRACK_INPUT}")

# Find Image_subsets directory (may be nested)
img_subsets = None
for d in sorted(WILDTRACK_INPUT.rglob("Image_subsets")):
    img_subsets = d
    break

if img_subsets is None:
    # Show what's available for debugging
    print("Contents of WILDTRACK dataset:")
    for p in sorted(WILDTRACK_INPUT.iterdir()):
        print(f"  {p.name}/ ({sum(1 for _ in p.iterdir()) if p.is_dir() else 'file'})")
    raise FileNotFoundError("Image_subsets not found in WILDTRACK dataset")

print(f"Image_subsets: {img_subsets}")
cameras = sorted([d.name for d in img_subsets.iterdir() if d.is_dir()])
print(f"Cameras ({len(cameras)}): {cameras}")

# Set up WILDTRACK directory
WILDTRACK_DIR = PROJECT / "data" / "raw" / "wildtrack"
VIDEOS_DIR = WILDTRACK_DIR / "videos"
VIDEOS_DIR.mkdir(parents=True, exist_ok=True)

# Generate MP4 videos from frame images
import cv2
for cam in cameras:
    cam_dir = img_subsets / cam
    frames = sorted(cam_dir.glob("*.png"))
    if not frames:
        frames = sorted(cam_dir.glob("*.jpg"))
    if not frames:
        print(f"  {cam}: No frames, skipping")
        continue

    out_path = VIDEOS_DIR / f"{cam}.mp4"
    if out_path.exists():
        print(f"  {cam}: Exists ({out_path.stat().st_size/1024**2:.1f} MB)")
        continue

    first = cv2.imread(str(frames[0]))
    h, w = first.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, 2.0, (w, h))
    for i, fp in enumerate(frames):
        writer.write(cv2.imread(str(fp)))
        if (i + 1) % 100 == 0:
            print(f"  {cam}: {i+1}/{len(frames)}")
    writer.release()
    print(f"  {cam}: {len(frames)} frames -> {out_path.name} ({w}x{h})")

# Symlink annotations and calibrations
for name in ["annotations_positions", "calibrations"]:
    src = None
    for d in WILDTRACK_INPUT.rglob(name):
        src = d
        break
    dst = WILDTRACK_DIR / name
    if src and not dst.exists():
        dst.symlink_to(src)
        print(f"Linked {name}")
    elif src:
        print(f"Already linked: {name}")
    else:
        print(f"WARNING: {name} not found")

print(f"\nVideos: {sorted(f.name for f in VIDEOS_DIR.glob('*.mp4'))}")

In [ ]:
import subprocess, sys, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
os.chdir(str(PROJECT))

result = subprocess.run(
    [sys.executable, "scripts/prepare_dataset.py", "--dataset", "wildtrack",
     "--root", "data/raw/wildtrack"],
    capture_output=True, text=True, cwd=str(PROJECT)
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError(f"GT preparation failed (exit code {result.returncode})")

gt_dir = PROJECT / "data" / "raw" / "wildtrack" / "manifests" / "ground_truth"
if gt_dir.exists():
    for f in sorted(gt_dir.glob("*")):
        if f.is_file():
            n = len(f.read_text().strip().split("\n"))
            print(f"  {f.name}: {n} lines")
else:
    print(f"WARNING: GT not found at {gt_dir}")

In [ ]:
import subprocess, time

os.chdir(str(PROJECT))

cmd = [
    sys.executable, "scripts/run_pipeline.py",
    "--config", "configs/default.yaml",
    "--dataset-config", "configs/datasets/wildtrack.yaml",
    "--stages", "0,1,2",
    "--override", f"project.run_name={RUN_NAME}",
    "--override", f"project.output_dir={DATA_OUT}",
    "--override", "stage1.detector.confidence_threshold=0.70",
    "--override", "stage1.tracker.min_hits=5",
    "--override", "stage1.tracker.track_high_thresh=0.70",
    "--override", "stage1.tracker.new_track_thresh=0.70",
    "--override", "stage1.tracker.track_buffer=120",
    "--override", "stage1.min_tracklet_length=8",
    "--override", "stage1.interpolation.max_gap=30",
    "--override", "stage1.intra_merge.max_time_gap=30.0",
    "--override", "stage2.pca.n_components=256",
    "--override", "stage2.camera_bn.enabled=true",
    "--override", "stage2.reid.flip_augment=true",
]

print(f"CMD: {' '.join(str(c) for c in cmd)}")
t0 = time.time()
result = subprocess.run(cmd, cwd=str(PROJECT))
elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min (exit={result.returncode})")

run_dir = DATA_OUT / RUN_NAME
for stage in ["stage0", "stage1", "stage2"]:
    sd = run_dir / stage
    if sd.exists():
        nf = sum(1 for _ in sd.rglob("*") if _.is_file())
        sz = sum(f.stat().st_size for f in sd.rglob("*") if f.is_file()) / 1024**2
        print(f"  {stage}: {nf} files ({sz:.1f} MB)")
    else:
        print(f"  {stage}: MISSING")

In [ ]:
import tarfile, json, os
from pathlib import Path

PROJECT = Path("/kaggle/working/gp")
DATA_OUT = Path("/tmp/pipeline_outputs")
OUTPUT = Path("/kaggle/working")

runs = sorted(DATA_OUT.glob("wildtrack_*"))
if not runs:
    raise FileNotFoundError("No wildtrack run directory found")
run_dir = runs[-1]
RUN_NAME = run_dir.name
print(f"Packaging: {RUN_NAME}")

metadata = {"run_name": RUN_NAME, "dataset": "wildtrack", "type": "person"}
meta_path = DATA_OUT / "run_metadata.json"
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

gt_dir = PROJECT / "data" / "raw" / "wildtrack" / "manifests" / "ground_truth"
ckpt_path = OUTPUT / "checkpoint.tar.gz"

with tarfile.open(str(ckpt_path), "w:gz") as tar:
    tar.add(str(meta_path), arcname="run_metadata.json")
    for stage in ["stage1", "stage2"]:
        sd = run_dir / stage
        if sd.exists():
            tar.add(str(sd), arcname=f"{RUN_NAME}/{stage}")
            print(f"  Added {stage}")
    manifest = run_dir / "stage0" / "frames_manifest.json"
    if manifest.exists():
        tar.add(str(manifest), arcname=f"{RUN_NAME}/stage0/frames_manifest.json")
        print("  Added stage0/frames_manifest.json")
    if gt_dir.exists():
        tar.add(str(gt_dir), arcname="gt_annotations")
        print("  Added gt_annotations")

sz = ckpt_path.stat().st_size / 1024**2
print(f"\nCheckpoint: {ckpt_path} ({sz:.1f} MB)")
summary = {"run_name": RUN_NAME, "dataset": "wildtrack", "stages": [0,1,2], "size_mb": round(sz,1)}
with open(OUTPUT / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))